# DeepCAVE-X 基线实验：SMAC / Optuna 轨迹生成 + DeepCAVE API 功能演示

> 这份 notebook 是 DeepCAVE-X 作业的**逻辑起点**。它只做两件事：
> 1. 用 **SMAC（贝叶斯优化）**、**Optuna-TPE（贝叶斯优化）** 和 **Optuna-Random（随机搜索基线）**
>    在公开分类数据集上运行 MLP 超参数优化，并通过 DeepCAVE `Recorder` 把轨迹写成原生 run；
> 2. 用 DeepCAVE 的 **API 模式**（不启动 Web 服务）重新加载这些 run，并调用多种插件生成分析图，
>    展示现有功能覆盖范围，方便后续在此基础上开发“多优化器轨迹对比 / 重要性 / 瓶颈诊断”新模块。

工作目录约定：

- `experiments/<dataset>/<optimizer_seed>/run`：DeepCAVE 原生 run 目录；
- `analysis_output/<dataset>/`：本 notebook 生成的插件图。

注意：本 notebook **不安装** SMAC / Optuna / DeepCAVE，请先在服务器环境配好后运行。

## 0. 运行环境

推荐环境（与任务计划一致）：

- Ubuntu 22.04（DeepCAVE 官方支持 Linux）
- Python 3.9 + DeepCAVE 1.4.1（含 history 加载补丁）
- `optuna`（版本建议与 DeepCAVE 的 `deepcave[optuna]` 对齐）、`smac3`、`ConfigSpace`
- `scikit-learn`、`pandas`、`matplotlib`/`plotly`（出图用 kaleido 或 HTML fallback）

在服务器上建议：

```bash
conda activate deepcave
python -m ipykernel install --user --name deepcave --display-name "DeepCAVE (3.9)"
```

然后用 VSCode 选择 **DeepCAVE (3.9)** kernel 运行本 notebook。

In [ ]:
# =============================
# 1. 实验规模参数（可自行调整）
# =============================

# 每个优化器每次运行允许的“完整配置”数（建议：调试 10-15，正式 50-100）
N_TRIALS = 15

# MLP 的 max_iter，同时充当 DeepCAVE 的多 fidelity budget。
# 同一组超参数会分别在 10 / 30 / 100 三个 budget 上评估，以覆盖 Budget Correlation 等插件。
BUDGETS = [10, 30, 100]

# 公开数据集（sklearn 内置，均来自 UCI/OpenML 等公开来源）
DATASETS = ["wine", "breast_cancer", "digits"]

# 每个 (dataset, optimizer) 的重复 seed 数；基线先跑 1 个，正式实验建议 3-5 个
SEEDS = [0]

# 数据划分 / 模型随机种子
DATA_SEED = 42
TEST_SIZE = 0.3

# 结果目录（相对当前 notebook 工作目录）
BASE_DIR = Path.cwd()
EXPERIMENT_ROOT = BASE_DIR / "experiments"
ANALYSIS_ROOT = BASE_DIR / "analysis_output"

# 重跑时覆盖旧 run（保证可重复）
OVERWRITE = True

for _p in (EXPERIMENT_ROOT, ANALYSIS_ROOT):
    _p.mkdir(parents=True, exist_ok=True)


In [ ]:
# =============================
# 2. 导入依赖
# =============================
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.datasets import load_wine, load_breast_cancer, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import balanced_accuracy_score

import ConfigSpace as CS
from ConfigSpace import Configuration

from deepcave import Recorder, Objective
from deepcave.runs.converters.deepcave import DeepCAVERun

# 插件类：后续功能演示会用到（部分插件耗时较长，这里只挑代表性插件）
from deepcave.plugins.objective.cost_over_time import CostOverTime
from deepcave.plugins.objective.pareto_front import ParetoFront
from deepcave.plugins.hyperparameter.parallel_coordinates import ParallelCoordinates
from deepcave.plugins.hyperparameter.pdp import PartialDependencies
from deepcave.plugins.hyperparameter.importances import Importances
from deepcave.plugins.hyperparameter.ablation_paths import AblationPaths

warnings.filterwarnings("ignore")

try:
    import optuna
    from optuna.samplers import TPESampler, RandomSampler
except ImportError as e:
    raise RuntimeError(
        "未找到 optuna。请在服务器 deepcave 环境中安装，例如 "
        "pip install 'deepcave[optuna]' optuna"
    ) from e

try:
    from smac import HyperparameterOptimizationFacade as HPOFacade
    from smac.scenario import Scenario
except ImportError as e:
    raise RuntimeError(
        "未找到 smac3。请在服务器环境中安装，例如 pip install smac3"
    ) from e

print("imports ok")


## 3. 公开数据集准备

默认使用三个 sklearn 内置公开分类数据集：`wine`、`breast_cancer`、`digits`。
它们体量小、分类标签明确，适合快速验证整条 DeepCAVE 数据链路；
如果希望换成 OpenML 数据集，只需在 `load_public_dataset` 中增加一个
`fetch_openml(...)` 分支即可（示例见注释）。

In [ ]:
def load_public_dataset(name: str):
    """加载公开分类数据集（X: ndarray, y: ndarray）。"""
    if name == "wine":
        return load_wine(return_X_y=True)
    if name == "breast_cancer":
        return load_breast_cancer(return_X_y=True)
    if name == "digits":
        return load_digits(return_X_y=True)

    # 如需 OpenML，可在此扩展，例如：
    # from sklearn.datasets import fetch_openml
    # df = fetch_openml(name="credit-g", as_frame=True, parser="auto")
    # return df.data.to_numpy(), (df.target == "good").astype(int).to_numpy()
    raise ValueError(f"Unknown dataset: {name}")


def prepare_data(name: str, seed: int = DATA_SEED):
    """固定分层划分 + 标准化，保证 SMAC/Optuna 使用完全相同的任务。"""
    X, y = load_public_dataset(name)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=seed
    )
    scaler = StandardScaler().fit(X_train)
    X_train = scaler.transform(X_train)
    X_test = scaler.transform(X_test)
    return {
        "name": name,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
    }


## 4. 搜索空间与目标函数

搜索空间覆盖连续（对数尺度）、整数、分类三类超参数；目标为双目标：

- `accuracy`（最大化）：测试集 balanced accuracy；
- `time`（最小化）：单次 fit + predict 的秒数。

保留双目标可以让 DeepCAVE 的 Pareto Front、Cost over Time 等多目标能力被后续使用。
同时每个配置在多个 `max_iter` budget 上评估，让 Budget Correlation 等 budget 插件也有数据。

In [ ]:
def make_configspace():
    """构造统一搜索空间：SMAC 与 Optuna 都使用同一份 ConfigSpace。"""
    configspace = CS.ConfigurationSpace(seed=0)
    configspace.add(
        [
            CS.hyperparameters.UniformFloatHyperparameter(
                name="alpha", lower=1e-6, upper=1e-1, log=True
            ),
            CS.hyperparameters.UniformFloatHyperparameter(
                name="learning_rate_init", lower=1e-4, upper=1e-1, log=True
            ),
            CS.hyperparameters.UniformIntegerHyperparameter(
                name="hidden_layer_sizes", lower=16, upper=128
            ),
            CS.hyperparameters.CategoricalHyperparameter(
                name="activation", choices=["relu", "tanh"]
            ),
            CS.hyperparameters.CategoricalHyperparameter(
                name="solver", choices=["adam", "sgd"]
            ),
            CS.hyperparameters.UniformIntegerHyperparameter(
                name="batch_size", lower=16, upper=256
            ),
        ]
    )
    return configspace


def make_objectives():
    """DeepCAVE 目标：accuracy 最大化 + time 最小化。"""
    return [
        Objective("accuracy", lower=0, upper=1, optimize="upper"),
        Objective("time", lower=0, optimize="lower"),
    ]


def evaluate_mlp(config, budget, prepared, seed):
    """对单一 (config, budget) 做一次 MLP 训练并返回 (accuracy, runtime)。"""
    params = {
        "hidden_layer_sizes": (int(config["hidden_layer_sizes"]),),
        "alpha": float(config["alpha"]),
        "learning_rate_init": float(config["learning_rate_init"]),
        "batch_size": int(config["batch_size"]),
        "activation": str(config["activation"]),
        "solver": str(config["solver"]),
    }
    t0 = time.perf_counter()
    clf = MLPClassifier(max_iter=int(budget), random_state=seed, **params)
    clf.fit(prepared["X_train"], prepared["y_train"])
    accuracy = float(
        balanced_accuracy_score(prepared["y_test"], clf.predict(prepared["X_test"]))
    )
    runtime = float(time.perf_counter() - t0)
    return accuracy, runtime


## 5. 通过 DeepCAVE `Recorder` 统一记录

`Recorder` 是本方案的“统一落盘层”：SMAC 与 Optuna 无论内部如何搜索，
最终每个被评估的配置都按 `(config, budget, seed)` 写入同一套原生 run 格式，
后续 `DeepCAVERun.from_path` 和插件都能无差别读取。

In [ ]:
def record_one_config(recorder, config, seed, prepared):
    """对一个配置按 BUDGETS 逐级评估并写入 DeepCAVE。

    返回 SMAC/Optuna 真正优化的标量：full-budget error = 1 - accuracy。
    """
    full_budget_error = None
    for budget in BUDGETS:
        recorder.start(config, budget=budget, seed=seed)
        accuracy, runtime = evaluate_mlp(config, budget, prepared, seed)
        recorder.end(costs=[accuracy, runtime], seed=seed)
        if budget == BUDGETS[-1]:
            full_budget_error = 1.0 - accuracy
    return full_budget_error


## 6. 运行 SMAC 与 Optuna

本步真正执行优化。SMAC 使用 BO（内部默认设计 + 随机森林代理），
Optuna 提供 TPE（BO）与 Random（随机基线）两种采样器。

> 调低 `N_TRIALS`、减少 `DATASETS` 可以缩短时间；正式实验再按需放大。

In [ ]:
def run_smac(dataset: str, seed: int, prepared: dict) -> Path:
    label = f"SMAC_seed{seed}"
    run_parent = EXPERIMENT_ROOT / dataset / label
    configspace = make_configspace()

    recorder_ref = {}

    def smac_target(config):
        return {"cost": record_one_config(recorder_ref["recorder"], config, seed, prepared)}

    with Recorder(
        configspace,
        objectives=make_objectives(),
        meta={
            "optimizer": "SMAC",
            "dataset": dataset,
            "seed": seed,
            "task": "baseline trajectory collection",
        },
        save_path=str(run_parent),
        prefix="run",
        overwrite=OVERWRITE,
    ) as recorder:
        recorder_ref["recorder"] = recorder

        scenario = Scenario(
            configspace=configspace,
            n_trials=N_TRIALS,
            n_workers=1,  # 保证 Recorder 只被单线程写入
            seed=seed,
            deterministic=True,
        )
        smac = HPOFacade(scenario, smac_target, overwrite=True)
        smac.optimize()
        recorded_path = recorder.path

    return Path(recorded_path)


def run_optuna(dataset: str, seed: int, sampler_name: str, prepared: dict) -> Path:
    label = f"Optuna_{sampler_name}_seed{seed}"
    run_parent = EXPERIMENT_ROOT / dataset / label
    configspace = make_configspace()
    recorder_ref = {}

    if sampler_name == "TPE":
        sampler = TPESampler(seed=seed)
    elif sampler_name == "Random":
        sampler = RandomSampler(seed=seed)
    else:
        raise ValueError(sampler_name)

    def optuna_objective(trial):
        values = {
            "alpha": trial.suggest_float("alpha", 1e-6, 1e-1, log=True),
            "learning_rate_init": trial.suggest_float(
                "learning_rate_init", 1e-4, 1e-1, log=True
            ),
            "hidden_layer_sizes": trial.suggest_int(
                "hidden_layer_sizes", 16, 128
            ),
            "activation": trial.suggest_categorical("activation", ["relu", "tanh"]),
            "solver": trial.suggest_categorical("solver", ["adam", "sgd"]),
            "batch_size": trial.suggest_int("batch_size", 16, 256),
        }
        config = Configuration(configspace, values=values)
        return record_one_config(recorder_ref["recorder"], config, seed, prepared)

    with Recorder(
        configspace,
        objectives=make_objectives(),
        meta={
            "optimizer": f"Optuna-{sampler_name}",
            "dataset": dataset,
            "seed": seed,
            "task": "baseline trajectory collection",
        },
        save_path=str(run_parent),
        prefix="run",
        overwrite=OVERWRITE,
    ) as recorder:
        recorder_ref["recorder"] = recorder

        study = optuna.create_study(direction="minimize", sampler=sampler)
        study.optimize(optuna_objective, n_trials=N_TRIALS)
        recorded_path = recorder.path

    return Path(recorded_path)


### 6.1 执行实验

下面的执行循环会依次跑 3 个数据集 × 3 种搜索策略。
默认参数下每个配置被评估 3 次 budget，因此总训练次数为：

`3 datasets × 3 strategies × N_TRIALS × len(BUDGETS)`。

In [ ]:
prepared_data = {}
run_paths = {}

for dataset in DATASETS:
    print(f"--- preparing dataset: {dataset} ---")
    prepared_data[dataset] = prepare_data(dataset, seed=DATA_SEED)

    for seed in SEEDS:
        # SMAC（贝叶斯优化）
        key_smac = (dataset, seed, "SMAC")
        run_paths[key_smac] = run_smac(dataset, seed, prepared_data[dataset])
        print("done:", key_smac)

        # Optuna：TPE 与 Random 两个采样器
        for sampler_name in ["TPE", "Random"]:
            key = (dataset, seed, f"Optuna_{sampler_name}")
            run_paths[key] = run_optuna(dataset, seed, sampler_name, prepared_data[dataset])
            print("done:", key)

print("\nall runs recorded")


## 7. 用 DeepCAVE API 重新加载并汇总

DeepCAVE 的核心 API 之一是 `DeepCAVERun.from_path(path)`。
这一节把所有 run 加载为对象并汇总基本统计，等价于 GUI 中 Overview 插件的数据侧。

In [ ]:
runs = {}
for key, path in run_paths.items():
    runs[key] = DeepCAVERun.from_path(path)

summary_rows = []
for (dataset, seed, strategy), run in runs.items():
    objectives = run.get_objectives()
    acc_obj = run.get_objective("accuracy")
    try:
        incumbent_config, incumbent_value = run.get_incumbent(objectives=acc_obj)
        incumbent_repr = incumbent_config.get_dictionary()
    except Exception as e:
        incumbent_value = None
        incumbent_repr = {"error": str(e)}

    summary_rows.append(
        {
            "dataset": dataset,
            "strategy": strategy,
            "seed": seed,
            "n_configs": run.get_num_configs(),
            "n_trials": len(run.history),
            "budgets": run.get_budget_ids(),
            "objectives": run.get_objective_names(),
            "best_accuracy": incumbent_value,
            "best_config": incumbent_repr,
            "path": str(run.path) if hasattr(run, "path") else str(run_paths[key]),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df


## 8. 插件 API 演示（现有 DeepCAVE 功能覆盖）

下面用 API 模式演示若干插件。每个插件统一遵循：

`plugin.generate_inputs(...) -> plugin.generate_outputs(run(s), inputs) -> plugin.load_outputs(...)`

为控制时间，插件图只对第一个演示数据集展示；切换 `DEMO_DATASET` 即可查看其他数据集。

In [ ]:
DEMO_DATASET = "digits" if "digits" in DATASETS else DATASETS[0]

demo_keys = {
    "SMAC": (DEMO_DATASET, SEEDS[0], "SMAC"),
    "Optuna_TPE": (DEMO_DATASET, SEEDS[0], "Optuna_TPE"),
    "Optuna_Random": (DEMO_DATASET, SEEDS[0], "Optuna_Random"),
}

demo_runs = {name: runs[key] for name, key in demo_keys.items()}
demo_smac = demo_runs["SMAC"]
print("demo dataset:", DEMO_DATASET)
print("SMAC objective ids:", demo_smac.get_objective_ids())
print("SMAC budget ids:", demo_smac.get_budget_ids())

accuracy_id = demo_smac.get_objective_ids()[0]
time_id = demo_smac.get_objective_ids()[1]
full_budget_id = demo_smac.get_budget_ids()[-1]
hp_names = list(demo_smac.configspace.keys())
print("hyperparameters:", hp_names)


def save_figure(fig, path: Path):
    """优先保存 PNG；若 kaleido 在服务器不可用则保存可交互 HTML。"""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        fig.write_image(str(path), width=1100, height=600, scale=1.2)
        print("saved:", path)
    except Exception as e:
        fallback = path.with_suffix(".html")
        fig.write_html(str(fallback))
        print(f"write_image failed ({e}); saved html: {fallback}")
    return path


### 8.1 Cost over Time：多条优化器轨迹对比

这个插件对应“轨迹对比”需求的基础视图，DeepCAVE 已支持一次传入多个 run。

In [ ]:
plugin = CostOverTime()
inputs = plugin.generate_inputs(
    objective_id=accuracy_id,
    budget_id=full_budget_id,
    xaxis="trials",
    show_runs=True,
    show_groups=True,
)

selected_runs = list(demo_runs.values())
outputs = plugin.generate_outputs(selected_runs, inputs)
fig_cot = plugin.load_outputs(selected_runs, inputs, outputs)

save_figure(fig_cot, ANALYSIS_ROOT / DEMO_DATASET / "cost_over_time.png")
fig_cot.show()


### 8.2 Pareto Front：双目标（accuracy vs time）

In [ ]:
plugin = ParetoFront()
inputs = plugin.generate_inputs(
    objective_id_1=accuracy_id,
    objective_id_2=time_id,
    budget_id=full_budget_id,
    show_all=True,
    show_error=False,
    show_runs=True,
    show_groups=True,
)

outputs = plugin.generate_outputs(selected_runs, inputs)
fig_pareto = plugin.load_outputs(selected_runs, inputs, outputs)

save_figure(fig_pareto, ANALYSIS_ROOT / DEMO_DATASET / "pareto_front.png")
fig_pareto.show()


### 8.3 Parallel Coordinates：配置与性能的关系

展示所有配置在超参数维度上的取值如何与 accuracy 关联。

In [ ]:
plugin = ParallelCoordinates()
inputs = plugin.generate_inputs(
    hyperparameter_names=hp_names,
    objective_id=accuracy_id,
    budget_id=full_budget_id,
    show_important_only=False,
    show_unsuccessful=False,
    hide_hps=False,
    n_hps=len(hp_names),
)

outputs = plugin.generate_outputs(demo_smac, inputs)
fig_parallel = plugin.load_outputs(demo_smac, inputs, outputs)

save_figure(fig_parallel, ANALYSIS_ROOT / DEMO_DATASET / "parallel_coordinates.png")
fig_parallel.show()


### 8.4 Importances：超参数重要性（fANOVA / LPI）

DeepCAVE 自带的 RF 代理 + fANOVA / LPI 是后续“重要性模块”可直接复用的基础。
这里用较少 `n_trees` 控制演示耗时。

In [ ]:
plugin = Importances()
inputs = plugin.generate_inputs(
    hyperparameter_names=hp_names,
    objective_id1=accuracy_id,
    objective_id2=None,  # None = 单目标；传 time_id 可看多目标重要性
    budget_ids=demo_smac.get_budget_ids(),
    method="global",  # 可选 local
    n_hps=len(hp_names),
    n_trees=30,
)

outputs = plugin.generate_outputs(demo_smac, inputs)
fig_importance = plugin.load_outputs(demo_smac, inputs, outputs)

save_figure(fig_importance, ANALYSIS_ROOT / DEMO_DATASET / "importances.png")
fig_importance.show()


### 8.5 Partial Dependencies：单/双超参数对性能的影响

In [ ]:
plugin = PartialDependencies()
inputs = plugin.generate_inputs(
    hyperparameter_name_1="learning_rate_init",
    hyperparameter_name_2="hidden_layer_sizes",
    objective_id=accuracy_id,
    budget_id=full_budget_id,
    show_confidence=False,
    show_ice=False,
)

outputs = plugin.generate_outputs(demo_smac, inputs)
fig_pdp = plugin.load_outputs(demo_smac, inputs, outputs)

save_figure(fig_pdp, ANALYSIS_ROOT / DEMO_DATASET / "partial_dependencies.png")
fig_pdp.show()


### 8.6 Ablation Paths：从默认配置到最优配置的改进路径

In [ ]:
plugin = AblationPaths()
inputs = plugin.generate_inputs(
    objective_id1=accuracy_id,
    objective_id2=None,
    budget_id=full_budget_id,
    n_hps=len(hp_names),
    n_trees=30,
    show_confidence=False,
)

outputs = plugin.generate_outputs(demo_smac, inputs)
figs = plugin.load_outputs(demo_smac, inputs, outputs)

for i, fig in enumerate(figs):
    if fig is not None:
        save_figure(fig, ANALYSIS_ROOT / DEMO_DATASET / f"ablation_paths_{i}.png")
        fig.show()


## 9. 当前覆盖范围与下一步开发入口

本 notebook 生成的数据与 API 演示覆盖了 DeepCAVE 主要能力中的：

| 能力 | 本 notebook 中的体现 |
|---|---|
| Run 数据模型 | `Recorder` + `DeepCAVERun.from_path` |
| 多 run / 多优化器 | SMAC、Optuna-TPE、Optuna-Random 三组 run |
| 多目标 | accuracy + time；Pareto Front 演示 |
| Multi-fidelity budget | 每个配置按 10/30/100 评估；budget 字段齐全 |
| 轨迹 | Cost over Time 多轨迹图 |
| 超参配置空间可视化 | Parallel Coordinates |
| 超参重要性 | Importances（fANOVA/LPI） |
| 依赖/响应面 | Partial Dependencies |
| 优化路径 | Ablation Paths |

尚未演示（GUI 侧为主，数据已支持）：Overview / Configurations / Configuration Footprint /
Configuration Cube / Budget Correlation / Symbolic Explanations / 自定义插件。

### DeepCAVE-X 新模块的接入点

1. **多优化器轨迹对比**：读取 `experiments/*/SMAC*/run`、`Optuna_*/run`，
   抽取 `run.get_trajectory(...)` 并对齐指标；
2. **重要性分析**：封装 `Importances`/fANOVA 结果，并支持后续加入 Sobol；
3. **瓶颈诊断**：对单条 run 的 incumbent 曲线做停滞窗口检测并在图上标注。

完成本 notebook 后，可基于 `experiments/` 中的 run 数据开始开发 `deepcave_x` 扩展包。